# 模块概述

WtShareHelper 是 WonderTrader 共享内存辅助模块，基于内存映射文件实现进程间共享内存。主要用于策略参数的共享和持久化，以及进程间命令传递。主要包括：
- 共享内存块管理（ShareBlocks）
- Master/Slave模式：Master创建和写入，Slave读取
- 三级数据结构：域（domain）、节（section）、键（key）
- 支持多种数据类型：int32、int64、uint32、uint64、double、string
- 命令队列功能，支持进程间命令传递
- C语言导出接口，支持跨语言调用

1. **接口层**（WtShareHelper.h/cpp）：
   - 提供C语言导出接口，支持Python、C#等外部语言调用
   - 封装ShareBlocks类的功能，提供简单的生命周期管理
   - 使用extern "C"确保C++代码可以被C语言调用
   - 提供Master和Slave模式的完整操作接口

2. **核心管理层**（ShareBlocks.h/cpp）：
   - ShareBlocks：共享内存块核心类，单例模式
   - 管理共享内存块的创建、打开、读写操作
   - 实现Master/Slave模式的控制逻辑
   - 提供域、节、键的三级结构管理
   - 支持多种数据类型的分配和读写
   - 提供命令队列功能，支持进程间命令传递

3. **数据结构层**（ShareBlocks.h）：
   - ShmBlock：共享内存块结构，包含标志、名称、节数组等
   - SecInfo：节信息结构，包含节名、键数组、数据区域等
   - KeyInfo：键信息结构，包含键名、类型、数据偏移、更新时间等
   - CmdBlock：命令块结构，实现命令队列功能
   - 使用紧凑的内存布局（#pragma pack），提高内存利用率

4. **底层支持**（BoostMappingFile）：
   - 基于Boost库的内存映射文件实现
   - 提供跨平台的内存映射文件操作
   - 支持文件的创建、打开、映射、同步等操作
   - 确保数据持久化和进程间共享

5. **工具支持**：
   - 提供回调函数类型定义（FuncGetSections、FuncGetKeys）
   - 支持数据有效性检查和同步机制
   - 提供时间戳管理，支持增量更新

# 层次关系图
```mermaid
graph LR
    %% 样式定义
    classDef interfaceClass fill:#e1f5ff,stroke:#01579b,stroke-width:3px,color:#000;
    classDef coreClass fill:#fff3e0,stroke:#e65100,stroke-width:3px,color:#000;
    classDef dataClass fill:#e8f5e9,stroke:#1b5e20,stroke-width:2px,color:#000;
    classDef baseClass fill:#f3e5f5,stroke:#4a148c,stroke-width:2px,color:#000;
    classDef externalClass fill:#f5f5f5,stroke:#616161,stroke-width:1px,stroke-dasharray: 5 5,color:#000;

    %% 外部接口层
    subgraph ExternalLayer["外部接口层 - C语言导出接口"]
        direction TB
        WtShareHelperC["WtShareHelper.h/cpp<br/>C语言导出接口<br/>• init_master/init_slave<br/>• allocate_xxx<br/>• set_xxx/get_xxx<br/>• commit_section<br/>• update_slave<br/>• init_cmder/add_cmd/get_cmd"]:::interfaceClass
    end

    %% 核心管理层
    subgraph CoreLayer["核心管理层 - 共享内存管理"]
        direction TB
        ShareBlocks["ShareBlocks<br/>共享内存块核心类<br/>• Master/Slave模式<br/>• 域-节-键管理<br/>• 数据分配和读写<br/>• 命令队列管理<br/>• 数据同步机制"]:::coreClass
    end

    %% 数据结构层
    subgraph DataLayer["数据结构层 - 内存布局"]
        direction TB
        ShmBlock["ShmBlock<br/>共享内存块结构<br/>• 标志和名称<br/>• 节数组（64个）<br/>• 更新时间戳"]:::dataClass
        SecInfo["SecInfo<br/>节信息结构<br/>• 节名称<br/>• 键数组（64个）<br/>• 数据区域（1024字节）"]:::dataClass
        KeyInfo["KeyInfo<br/>键信息结构<br/>• 键名称<br/>• 数据类型<br/>• 数据偏移<br/>• 更新时间"]:::dataClass
        CmdBlock["CmdBlock<br/>命令块结构<br/>• 命令队列<br/>• 读写索引<br/>• 命令数据"]:::dataClass
    end

    %% 底层支持
    subgraph BaseLayer["底层支持"]
        direction TB
        BoostMappingFile["BoostMappingFile<br/>内存映射文件<br/>• 文件创建/打开<br/>• 内存映射<br/>• 数据同步<br/>• 跨平台支持"]:::baseClass
    end

    %% 外部系统
    subgraph ExternalSys["外部系统"]
        direction TB
        MasterProcess["Master进程<br/>• 创建共享内存<br/>• 写入数据<br/>• 发送命令"]:::externalClass
        SlaveProcess["Slave进程<br/>• 连接共享内存<br/>• 读取数据<br/>• 接收命令"]:::externalClass
    end

    %% 组合关系
    WtShareHelperC -->|"委托给"| ShareBlocks
    ShareBlocks -->|"管理"| ShmBlock
    ShmBlock -->|"包含"| SecInfo
    SecInfo -->|"包含"| KeyInfo
    ShmBlock -->|"包含"| CmdBlock
    ShareBlocks -->|"使用"| BoostMappingFile

    %% 数据访问关系
    ShareBlocks -->|"读写"| ShmBlock
    ShareBlocks -->|"分配/设置"| KeyInfo
    ShareBlocks -->|"提交"| SecInfo
    ShareBlocks -->|"管理命令"| CmdBlock

    %% 进程间关系
    MasterProcess -->|"调用"| WtShareHelperC
    WtShareHelperC -->|"写入"| ShareBlocks
    ShareBlocks -->|"映射到文件"| BoostMappingFile
    BoostMappingFile -->|"共享内存"| SlaveProcess
    SlaveProcess -->|"调用"| WtShareHelperC
    WtShareHelperC -->|"读取"| ShareBlocks

    %% 应用样式
    class WtShareHelperC interfaceClass
    class ShareBlocks coreClass
    class ShmBlock,SecInfo,KeyInfo,CmdBlock dataClass
    class BoostMappingFile baseClass
    class MasterProcess,SlaveProcess externalClass
```

# C语言导出接口 WtShareHelper.h/cpp

## 初始化与生命周期管理

### 初始化Master模式 init_master
```cpp
/**
 * @brief 初始化Master模式的C接口实现
 * @param id 域ID
 * @param path 文件路径
 * @return 返回初始化是否成功
 * 
 * 调用ShareBlocks的init_master方法初始化Master模式。
 */
bool init_master(const char* id, const char* path/* = ""*/)
{
	return ShareBlocks::one().init_master(id, path);  // 调用ShareBlocks的init_master方法
}
```

### 初始化Slave模式 init_slave
```cpp
/**
 * @brief 初始化Slave模式的C接口实现
 * @param id 域ID
 * @param path 文件路径
 * @return 返回初始化是否成功
 * 
 * 调用ShareBlocks的init_slave方法初始化Slave模式。
 */
bool init_slave(const char* id, const char* path/* = ""*/)
{
	return ShareBlocks::one().init_slave(id, path);   // 调用ShareBlocks的init_slave方法
}
```

### 更新Slave数据 update_slave
```cpp
/**
 * @brief 更新Slave数据的C接口实现
 * @param id 域ID
 * @param bForce 是否强制更新
 * @return 返回是否更新成功
 * 
 * 调用ShareBlocks的update_slave方法更新Slave数据。
 */
bool update_slave(const char* id, bool bForce/* = false*/)
{
	return ShareBlocks::one().update_slave(id, bForce);  // 调用ShareBlocks的update_slave方法
}
```

### 释放Slave连接 release_slave
```cpp
/**
 * @brief 释放Slave连接的C接口实现
 * @param name 域ID
 * @return 返回是否释放成功
 * 
 * 调用ShareBlocks的release_slave方法释放Slave连接。
 */
bool release_slave(const char* name)
{
	return ShareBlocks::one().release_slave(name);     // 调用ShareBlocks的release_slave方法
}
```

## 节管理

### 获取节列表 get_sections
```cpp
/**
 * @brief 获取节列表的C接口实现
 * @param domain 域名称
 * @param cb 回调函数指针
 * @return 返回节的数量
 * 
 * 获取节列表，通过回调函数返回每个节的名称。
 */
uint32_t get_sections(const char* domain, FuncGetSections cb)
{
	auto ay = ShareBlocks::one().get_sections(domain);  // 调用ShareBlocks的get_sections方法，获取节名称向量
	for (const std::string& v : ay)                     // 遍历节名称向量
		cb(v.c_str());                                  // 调用回调函数，传递节名称

	return (uint32_t)ay.size();                         // 返回节的数量
}
```

### 获取键列表 get_keys
```cpp
/**
 * @brief 获取键列表的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param cb 回调函数指针
 * @return 返回键的数量
 * 
 * 获取键列表，通过回调函数返回每个键的名称和类型。
 */
uint32_t get_keys(const char* domain, const char* section, FuncGetKeys cb)
{
	auto ay = ShareBlocks::one().get_keys(domain, section);  // 调用ShareBlocks的get_keys方法，获取键信息向量
	for (KeyInfo* v : ay)                                    // 遍历键信息向量
		cb(v->_key, v->_type);                               // 调用回调函数，传递键名称和类型

	return (uint32_t)ay.size();                              // 返回键的数量
}
```

### 获取节的更新时间 get_section_updatetime
```cpp
/**
 * @brief 获取节的更新时间的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @return 返回更新时间戳
 * 
 * 调用ShareBlocks的get_section_updatetime方法获取节的更新时间。
 */
uint64_t get_section_updatetime(const char* domain, const char* section)
{
	return ShareBlocks::one().get_section_updatetime(domain, section);  // 调用ShareBlocks的get_section_updatetime方法
}
```

### 提交节（更新修改时间）commit_section
```cpp
/**
 * @brief 提交节的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @return 返回是否提交成功
 * 
 * 调用ShareBlocks的commit_section方法提交节。
 */
bool commit_section(const char* domain, const char* section)
{
	return ShareBlocks::one().commit_section(domain, section);  // 调用ShareBlocks的commit_section方法
}
```

### 删除节 delete_section
```cpp
/**
 * @brief 删除节的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @return 返回是否删除成功
 * 
 * 调用ShareBlocks的delete_section方法删除节。
 */
bool delete_section(const char* domain, const char*section)
{
	return ShareBlocks::one().delete_section(domain, section);   // 调用ShareBlocks的delete_section方法
}
```

## 键值对分配

### 分配字符串类型的键值对 allocate_string
```cpp
/**
 * @brief 分配字符串类型键值对的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param initVal 初始值
 * @param bForceWrite 是否强制写入
 * @return 返回字符串指针
 * 
 * 调用ShareBlocks的allocate_string方法分配字符串类型键值对。
 */
const char* allocate_string(const char* domain, const char* section, const char* key, const char* initVal, bool bForceWrite /*= false*/)
{
	return ShareBlocks::one().allocate_string(domain, section, key, initVal, bForceWrite);  // 调用ShareBlocks的allocate_string方法
}
```

### 分配int32类型的键值对 allocate_int32
```cpp
/**
 * @brief 分配int32类型键值对的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param initVal 初始值
 * @param bForceWrite 是否强制写入
 * @return 返回int32指针
 * 
 * 调用ShareBlocks的allocate_int32方法分配int32类型键值对。
 */
int32_t* allocate_int32(const char* domain, const char* section, const char* key, int32_t initVal, bool bForceWrite /*= false*/)
{
	return ShareBlocks::one().allocate_int32(domain, section, key, initVal, bForceWrite);  // 调用ShareBlocks的allocate_int32方法
}
```

### 分配int64类型的键值对 allocate_int64
```cpp
/**
 * @brief 分配int64类型键值对的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param initVal 初始值
 * @param bForceWrite 是否强制写入
 * @return 返回int64指针
 * 
 * 调用ShareBlocks的allocate_int64方法分配int64类型键值对。
 */
int64_t* allocate_int64(const char* domain, const char* section, const char* key, int64_t initVal, bool bForceWrite /*= false*/)
{
	return ShareBlocks::one().allocate_int64(domain, section, key, initVal, bForceWrite);  // 调用ShareBlocks的allocate_int64方法
}
```

### 分配uint32类型的键值对 allocate_uint32
```cpp
/**
 * @brief 分配uint32类型键值对的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param initVal 初始值
 * @param bForceWrite 是否强制写入
 * @return 返回uint32指针
 * 
 * 调用ShareBlocks的allocate_uint32方法分配uint32类型键值对。
 */
uint32_t* allocate_uint32(const char* domain, const char* section, const char* key, uint32_t initVal, bool bForceWrite /*= false*/)
{
	return ShareBlocks::one().allocate_uint32(domain, section, key,  initVal, bForceWrite);  // 调用ShareBlocks的allocate_uint32方法
}
```

### 分配uint64类型的键值对 allocate_uint64
```cpp
/**
 * @brief 分配uint64类型键值对的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param initVal 初始值
 * @param bForceWrite 是否强制写入
 * @return 返回uint64指针
 * 
 * 调用ShareBlocks的allocate_uint64方法分配uint64类型键值对。
 */
uint64_t* allocate_uint64(const char* domain, const char* section, const char* key, uint64_t initVal, bool bForceWrite /*= false*/)
{
	return ShareBlocks::one().allocate_uint64(domain, section, key, initVal, bForceWrite);  // 调用ShareBlocks的allocate_uint64方法
}
```

### 分配double类型的键值对 allocate_double
```cpp
/**
 * @brief 分配double类型键值对的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param initVal 初始值
 * @param bForceWrite 是否强制写入
 * @return 返回double指针
 * 
 * 调用ShareBlocks的allocate_double方法分配double类型键值对。
 */
double* allocate_double(const char* domain, const char* section, const char* key, double initVal, bool bForceWrite /*= false*/)
{
	return ShareBlocks::one().allocate_double(domain, section, key, initVal, bForceWrite);  // 调用ShareBlocks的allocate_double方法
}
```

## 键值对设置

### 设置字符串值 set_string
```cpp
/**
 * @brief 设置字符串值的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param val 值
 * @return 返回是否设置成功
 * 
 * 调用ShareBlocks的set_string方法设置字符串值。
 */
bool set_string(const char* domain, const char* section, const char* key, const char* val)
{
	return ShareBlocks::one().set_string(domain, section, key, val);  // 调用ShareBlocks的set_string方法
}
```

### 设置int32值 set_int32
```cpp
/**
 * @brief 设置int32值的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param val 值
 * @return 返回是否设置成功
 * 
 * 调用ShareBlocks的set_int32方法设置int32值。
 */
bool set_int32(const char* domain, const char* section, const char* key, int32_t val)
{
	return ShareBlocks::one().set_int32(domain, section, key, val);  // 调用ShareBlocks的set_int32方法
}
```

### 设置int64值 set_int64
```cpp
/**
 * @brief 设置int64值的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param val 值
 * @return 返回是否设置成功
 * 
 * 调用ShareBlocks的set_int64方法设置int64值。
 */
bool set_int64(const char* domain, const char* section, const char* key, int64_t val)
{
	return ShareBlocks::one().set_int64(domain, section, key, val);  // 调用ShareBlocks的set_int64方法
}
```

### 设置uint32值 set_uint32
```cpp
/**
 * @brief 设置uint32值的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param val 值
 * @return 返回是否设置成功
 * 
 * 调用ShareBlocks的set_uint32方法设置uint32值。
 */
bool set_uint32(const char* domain, const char* section, const char* key, uint32_t val)
{
	return ShareBlocks::one().set_uint32(domain, section, key, val);  // 调用ShareBlocks的set_uint32方法
}
```

### 设置uint64值 set_uint64
```cpp
/**
 * @brief 设置uint64值的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param val 值
 * @return 返回是否设置成功
 * 
 * 调用ShareBlocks的set_uint64方法设置uint64值。
 */
bool set_uint64(const char* domain, const char* section, const char* key, uint64_t val)
{
	return ShareBlocks::one().set_uint64(domain, section, key, val);  // 调用ShareBlocks的set_uint64方法
}
```

### 设置double值 set_double
```cpp
/**
 * @brief 设置double值的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param val 值
 * @return 返回是否设置成功
 * 
 * 调用ShareBlocks的set_double方法设置double值。
 */
bool set_double(const char* domain, const char* section, const char* key, double val)
{
	return ShareBlocks::one().set_double(domain, section, key, val);  // 调用ShareBlocks的set_double方法
}
```

## 键值对获取

### 获取字符串值 get_string
```cpp
/**
 * @brief 获取字符串值的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param defVal 默认值
 * @return 返回字符串指针
 * 
 * 调用ShareBlocks的get_string方法获取字符串值。
 */
const char* get_string(const char* domain, const char* section, const char* key, const char* defVal /* = "" */)
{
	return ShareBlocks::one().get_string(domain, section, key, defVal);  // 调用ShareBlocks的get_string方法
}
```

### 获取int32值 get_int32
```cpp
/**
 * @brief 获取int32值的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param defVal 默认值
 * @return 返回int32值
 * 
 * 调用ShareBlocks的get_int32方法获取int32值。
 */
int32_t get_int32(const char* domain, const char* section, const char* key, int32_t defVal /* = 0 */)
{
	return ShareBlocks::one().get_int32(domain, section, key, defVal);  // 调用ShareBlocks的get_int32方法
}
```

### 获取int64值 get_int64
```cpp
/**
 * @brief 获取int64值的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param defVal 默认值
 * @return 返回int64值
 * 
 * 调用ShareBlocks的get_int64方法获取int64值。
 */
int64_t get_int64(const char* domain, const char* section, const char* key, int64_t defVal /* = 0 */)
{
	return ShareBlocks::one().get_int64(domain, section, key, defVal);  // 调用ShareBlocks的get_int64方法
}
```

### 获取uint32值 get_uint32
```cpp
/**
 * @brief 获取uint32值的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param defVal 默认值
 * @return 返回uint32值
 * 
 * 调用ShareBlocks的get_uint32方法获取uint32值。
 */
uint32_t get_uint32(const char* domain, const char* section, const char* key, uint32_t defVal /* = 0 */)
{
	return ShareBlocks::one().get_uint32(domain, section, key, defVal);  // 调用ShareBlocks的get_uint32方法
}
```

### 获取uint64值 get_uint64
```cpp
/**
 * @brief 获取uint64值的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param defVal 默认值
 * @return 返回uint64值
 * 
 * 调用ShareBlocks的get_uint64方法获取uint64值。
 */
uint64_t get_uint64(const char* domain, const char* section, const char* key, uint64_t defVal /* = 0 */)
{
	return ShareBlocks::one().get_uint64(domain, section, key, defVal);  // 调用ShareBlocks的get_uint64方法
}
```

### 获取double值 get_double
```cpp
/**
 * @brief 获取double值的C接口实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param defVal 默认值
 * @return 返回double值
 * 
 * 调用ShareBlocks的get_double方法获取double值。
 */
double get_double(const char* domain, const char* section, const char* key, double defVal /* = 0 */)
{
	return ShareBlocks::one().get_double(domain, section, key, defVal);  // 调用ShareBlocks的get_double方法
}
```

## 命令队列管理

### 初始化命令队列 init_cmder
```cpp
/**
 * @brief 初始化命令队列的C接口实现
 * @param name 命令队列名称
 * @param isCmder 是否为命令下达者
 * @param path 文件路径
 * @return 返回初始化是否成功
 * 
 * 调用ShareBlocks的init_cmder方法初始化命令队列。
 */
bool init_cmder(const char* name, bool isCmder /* = false */, const char* path /* = "" */)
{
	return ShareBlocks::one().init_cmder(name, isCmder, path);  // 调用ShareBlocks的init_cmder方法
}
```

### 添加命令 add_cmd
```cpp
/**
 * @brief 添加命令的C接口实现
 * @param name 命令队列名称
 * @param cmd 命令内容
 * @return 返回是否添加成功
 * 
 * 调用ShareBlocks的add_cmd方法添加命令。
 */
bool add_cmd(const char* name, const char* cmd)
{
	return ShareBlocks::one().add_cmd(name, cmd);  // 调用ShareBlocks的add_cmd方法
}
```

### 获取命令 get_cmd
```cpp
/**
 * @brief 获取命令的C接口实现
 * @param name 命令队列名称
 * @param lastIdx 上次读取的索引（引用）
 * @return 返回命令内容
 * 
 * 调用ShareBlocks的get_cmd方法获取命令。
 */
const char* get_cmd(const char* name, uint32_t& lastIdx)
{
	return ShareBlocks::one().get_cmd(name, lastIdx);  // 调用ShareBlocks的get_cmd方法
}
```

# 共享内存块核心类 ShareBlocks.h/cpp
```cpp
class ShareBlocks
```
实现基于内存映射文件的进程间共享内存。
- 使用内存映射文件（Memory-Mapped File）实现进程间共享内存
- 支持Master/Slave模式：Master创建和写入，Slave读取
- 提供域（domain）、节（section）、键（key）的三级结构
- 支持多种数据类型：int32、int64、uint32、uint64、double、string
- 使用紧凑的内存布局（#pragma pack），提高内存利用率

## 成员
- `ShmBlockMap _shm_blocks`：共享内存块映射表
  - typedef wt_hashmap\<std::string, `ShmPair`\> ShmBlockMap：共享内存块映射表
    ```cpp
    /**
     * @struct ShmPair
    * @brief 共享内存块配对结构，存储一个共享内存块的所有相关信息
    */
    typedef struct _ShmPair
    {
      MappedFilePtr _domain; // 内存映射文件智能指针，管理内存映射文件的生命周期
      bool _master; // Master标志（布尔值），true表示Master模式，false表示Slave模式
      uint64_t _blocktime; // 块的缓存时间戳（64位无符号整数，毫秒），用于检测数据是否变化

      ShmBlock* _block; // 共享内存块指针，指向映射的内存区域
      SectionMap _sections; // 节映射表，快速查找节和键
    }ShmPair;

- `CmdBlockMap _cmd_blocks`：命令块映射表
  - typedef wt_hashmap\<std::string, `CmdPair`\> CmdBlockMap：命令块映射表类型定义
    ```cpp
    /**
     * @struct CmdPair
    * @brief 命令块配对结构
    */
    typedef struct _CmdPair
    {
      MappedFilePtr _domain; // 内存映射文件智能指针，管理内存映射文件的生命周期
      CmdBlock* _block; // 命令块指针，指向映射的内存区域
      bool _cmder; // 命令下达者标志（布尔值），true表示下达命令，false表示接收命令
    } CmdPair;
    ```

## 获取单例实例 one
```cpp
/**
 * @brief 获取单例实例
 * @return 返回ShareBlocks实例的引用
 * 
 * 使用Meyers' Singleton模式实现单例。
 * 线程安全（C++11标准保证静态局部变量初始化的线程安全性）。
 */
static ShareBlocks& one()
{
    static ShareBlocks inst; // 静态局部变量：全局唯一的ShareBlocks实例，首次调用时初始化
    return inst; // 返回实例的引用
}
```

## 初始化与生命周期管理

### 初始化Master模式 init_master
初始化共享内存的**写端（Master）**。负责创建物理文件、建立内存映射、清理无效数据以及构建内存索引。
* **检查是否已初始化**：
  * 通过 `name` 在 `_shm_blocks` 中查找。
  * 如果 `shm._block` 不为空，说明已初始化，直接返回 `true`。
* **物理文件准备**：
  * 确定文件名：如果 `path` 为空，则使用 `name`。
  * **创建文件**：如果文件不存在，使用 `BoostFile` 创建新文件，并将其大小截断（truncate）为 `sizeof(ShmBlock)`，然后关闭文件。
* **内存映射**：
  * 创建 `BoostMappingFile` 对象。
  * 将文件映射到内存。
  * 标记当前实例为 Master (`shm._master = true`)。
  * 获取映射地址并转换为 `ShmBlock*` 指针，置给 `shm._block`
* **清理无效节**：
  * 遍历共享内存块中的所有节 `shm._block->_sections`
  * **筛选有效节**：将 `_count > 0` 且 `_state == 1` 的节暂存到 `aySecs` 向量中。
  * **重组内存**：如果发现有无效节（有效数量 != 总数量）：
    * 更新 `shm._block->_count` 为有效数量。
    * 清空整个 `shm._block->_sections` 内存区域
    * 将筛选出的有效节数据拷贝回 `shm._block->_sections`
    * 更新块的 `shm._blocktime = shm._block->_updatetime`
* **构建索引**：
  * 再次遍历整理后的节 `shm._block->_sections`：
    * 如果节内没有键 (`_count == 0`)，跳过。
    * **重置状态**：将节的 `_state` 置为 0
      * 根据注释，用于后续检测该节在本次运行中是否被使用/分配，若未被使用下次启动可能会被清理
    * **建立映射**：
      * `shm._sections[shm._block->_sections[i]._name]._index = i`
      * 遍历节内的所有 Key `shm._block->_sections[i]._keys`
        * `shm._sections[shm._block->_sections[i]._name]._keys[shm._block->_sections[i]._keys[j]._key] = &shm._block->_sections[i]._keys[j]` 

```cpp
/**
 * @brief 初始化Master模式的实现
 * @param name 域名称
 * @param path 文件路径
 * @return 返回初始化是否成功
 */
bool ShareBlocks::init_master(const char* name, const char* path/* = ""*/)
```

### 初始化Slave模式 init_slave
初始化共享内存的**读端（Slave）**。负责连接已存在的共享内存文件，并构建用于读取的内存索引。
* **检查是否已初始化**：
  * 如果 `shm._block` 不为空，直接返回 `true`。
* **物理文件检查**：
  * 确定文件名。
  * **存在性检查**：如果文件不存在，直接返回 `false`（Slave 模式无权创建文件）。
* **内存映射**：
  * 创建 `BoostMappingFile` 对象并映射文件。
  * 标记当前实例为 Slave (`shm._master = false`)。
  * 获取映射地址并转换为 `ShmBlock*` 指针，置给 `shm._block`
  * **记录时间戳**：保存当前块的更新时间 `_updatetime` 到本地 `shm._blocktime`，用于后续检测数据变动。
* **构建索引**：
  * 遍历共享内存块中的所有节 `shm._block->_sections`：
    * **跳过无效节**：如果 `_count == 0` 或 `_state != 1`，则忽略。
    * **建立映射**：
      * `shm._sections[shm._block->_sections[i]._name]._index = i`
      * 遍历节内的所有 Key `shm._block->_sections[i]._keys`
        * `shm._sections[shm._block->_sections[i]._name]._keys[shm._block->_sections[i]._keys[j]._key] = &shm._block->_sections[i]._keys[j]` 

```cpp
/**
 * @brief 初始化Slave模式的实现
 * @param name 域名称
 * @param path 文件路径
 * @return 返回初始化是否成功
 */
bool ShareBlocks::init_slave(const char* name, const char* path/* = ""*/)
```

### 更新Slave数据 update_slave
当 Master 端修改了共享内存结构（如新增了 Key 或 Section）时，Slave 端通过此函数**刷新本地索引**，以感知最新的内存布局。
* **前置检查**：
  * 如果共享内存块指针 `_shm_blocks[name]._block` 空，返回 `false`。
* **检查更新必要性**：
  * 比较本地记录的时间戳 `shm._blocktime` 与共享内存块的实际时间戳 `shm._block->_updatetime`。
  * 如果时间戳一致（数据未变动）**且** `!bForce`（非强制更新），则返回 `false`。
* **重建索引**：
  * **清空旧索引**：调用 `shm._sections.clear()` 清除当前的哈希映射。
  * **重新加载：遍历`shm._block->_sections`**：
    * 跳过空节（`_count == 0`）
    * **更新映射**：
      * `shm._sections[shm._block->_sections[i]._name]._index = i`
      * 遍历节内的所有 Key `shm._block->_sections[i]._keys`
        * `shm._sections[shm._block->_sections[i]._name]._keys[shm._block->_sections[i]._keys[j]._key] = &shm._block->_sections[i]._keys[j]` 
* **更新状态**：
  * 更新本地 `shm._blocktime` 为最新的 `shm._block->_updatetime`。
  * 返回 `true`。
```cpp
/**
 * @brief 更新Slave数据的实现
 * @param name 域名称
 * @param bForce 是否强制更新
 * @return 返回是否更新成功
 */
bool ShareBlocks::update_slave(const char* name, bool bForce)
```

### 释放Slave连接 release_slave
断开与共享内存的连接，清理本地维护的资源。通常用于程序退出或不再需要访问某块共享内存时。
* **查找实例**：
  * 在 `_shm_blocks` 中查找指定 `name` 的共享块。如果不包含该块，视为已释放，返回 `true`。
* **权限检查**：
  * **Master 保护**：如果当前是 Master 模式 (`shm._master` 为 true)，则不允许通过此接口释放，返回 `false`。
* **资源清理**：
  * 置空块指针 `shm._block = NULL`。
  * 清空节索引映射表 `shm._sections.clear()`。
  * **释放文件映射**：重置智能指针 `shm._domain.reset()`，这将触发 `BoostMappingFile` 的析构，自动解除内存映射并关闭文件句柄。
  * 重置时间戳 `shm._blocktime = 0`。
* **移除记录**：
  * 从全局管理 Map `_shm_blocks` 中擦除（erase）该条目。
  * 返回 `true`。
```cpp
/**
 * @brief 释放Slave连接的实现
 * @param name 域名称
 * @return 返回是否释放成功
 */
bool ShareBlocks::release_slave(const char* name)
```

## 节管理

### 获取节列表 get_sections
```cpp
/**
 * @brief 获取节列表的实现
 * @param domain 域名称
 * @return 返回节名称向量
 * 
 * 获取流程：
 * 1. 查找共享内存块
 * 2. 遍历所有节，收集状态为1（生效）的节名称
 * 3. 返回节名称向量
 */
std::vector<std::string> ShareBlocks::get_sections(const char* domain)
{
	static std::vector<std::string> emptyRet;    // 静态空向量，用于返回空结果

	auto it = _shm_blocks.find(domain);          // 在映射表中查找共享内存块
	if (it == _shm_blocks.end())                 // 如果不存在，返回空向量
		return emptyRet;

	std::vector<std::string> ret;                // 结果向量
	const ShmPair& shm = it->second;             // 获取共享内存块配对引用
	for (uint32_t i = 0; i < shm._block->_count; i++)  // 遍历所有节
	{
		if (shm._block->_sections[i]._state != 1)  // 如果节状态不为1（无效或已删除）
			continue;                            // 跳过该节

		ret.emplace_back(shm._block->_sections[i]._name);  // 将节名称添加到结果向量
	}

	return std::move(ret);                        // 移动返回结果向量（避免拷贝）
}
```

### 获取键列表 get_keys
```cpp
/**
 * @brief 获取键列表的实现
 * @param domain 域名称
 * @param section 节名称
 * @return 返回键信息指针向量
 * 
 * 获取流程：
 * 1. 查找共享内存块
 * 2. 如果索引映射表与块中的节数量不一致，强制更新Slave
 * 3. 查找节
 * 4. 收集节中所有键的信息指针
 * 5. 返回键信息指针向量
 */
std::vector<KeyInfo*> ShareBlocks::get_keys(const char* domain, const char* section)
{
	static std::vector<KeyInfo*> emptyRet;       // 静态空向量，用于返回空结果

	auto it = _shm_blocks.find(domain);          // 在映射表中查找共享内存块
	if (it == _shm_blocks.end())                 // 如果不存在，返回空向量
		return emptyRet;

	const ShmPair& shm = it->second;             // 获取共享内存块配对引用
	if(shm._sections.size() != shm._block->_count)  // 如果索引映射表与块中的节数量不一致（数据可能已更新）
	{
		update_slave(domain, true);              // 强制更新Slave，重新加载数据
	}

	auto sit = shm._sections.find(section);      // 在节的映射表中查找节
	if (sit == shm._sections.end())              // 如果不存在，返回空向量
		return emptyRet;

	std::vector<KeyInfo*> ret;                   // 结果向量
	const ShmPair::KVPair& kvPair = sit->second;  // 获取键值对引用
	for (auto& v : kvPair._keys)                 // 遍历键映射表
	{
		ret.emplace_back(v.second);              // 将键信息指针添加到结果向量
	}

	return std::move(ret);                       // 移动返回结果向量（避免拷贝）
}
```

### 获取节的更新时间 get_section_updatetime
```cpp
/**
 * @brief 获取节的更新时间的实现
 * @param domain 域名称
 * @param section 节名称
 * @return 返回更新时间戳
 * 
 * 获取流程：
 * 1. 查找共享内存块
 * 2. 查找节
 * 3. 返回节的更新时间戳
 */
uint64_t ShareBlocks::get_section_updatetime(const char* domain, const char* section)
{
	auto it = _shm_blocks.find(domain);          // 在映射表中查找共享内存块
	if (it == _shm_blocks.end())                 // 如果不存在，返回0
		return 0;

	const ShmPair& shm = (ShmPair&)it->second;  // 获取共享内存块配对引用
	auto sit = shm._sections.find(section);      // 在节的映射表中查找节
	if (sit == shm._sections.end())             // 如果不存在，返回0
		return 0;

	const ShmPair::KVPair& kvPair = sit->second;  // 获取键值对引用
	const SecInfo& secInfo = shm._block->_sections[kvPair._index];  // 获取节信息
	return secInfo._updatetime;                  // 返回节的更新时间戳
}
```

### 提交节（更新修改时间）commit_section
```cpp
/**
 * @brief 提交节的实现
 * @param domain 域名称
 * @param section 节名称
 * @return 返回是否提交成功
 * 
 * 提交流程：
 * 1. 查找共享内存块和节
 * 2. 更新节的修改时间为当前时间
 * 3. 这会触发块的更新时间戳变化，通知Slave数据已更新
 */
bool ShareBlocks::commit_section(const char* domain, const char* section)
{
	auto it = _shm_blocks.find(domain);          // 在映射表中查找共享内存块
	if (it == _shm_blocks.end())                 // 如果不存在，返回失败
		return false;

	ShmPair& shm = (ShmPair&)it->second;         // 获取共享内存块配对引用
	auto sit = shm._sections.find(section);      // 在节的映射表中查找节
	if (sit == shm._sections.end())              // 如果不存在，返回失败
		return false;

	ShmPair::KVPair& kvPair = (ShmPair::KVPair&)sit->second;  // 获取键值对引用
	SecInfo& secInfo = shm._block->_sections[kvPair._index];  // 获取节信息引用
	secInfo._updatetime = TimeUtils::getLocalTimeNow();  // 更新节的修改时间为当前时间（毫秒）
	return true;                                 // 返回成功
}
```

### 删除节 delete_section
```cpp
/**
 * @brief 删除节的实现
 * @param domain 域名称
 * @param section 节名称
 * @return 返回是否删除成功
 * 
 * 删除流程：
 * 1. 查找共享内存块和节
 * 2. 从索引映射表中移除节
 * 3. 将节的state标记为2（已删除）
 * 4. 更新块的修改时间
 */
bool ShareBlocks::delete_section(const char* domain, const char*section)
{
	auto it = _shm_blocks.find(domain);          // 在映射表中查找共享内存块
	if (it == _shm_blocks.end())                 // 如果不存在，返回失败
		return false;

	ShmPair& shm = (ShmPair&)it->second;         // 获取共享内存块配对引用
	auto sit = shm._sections.find(section);      // 在节的映射表中查找节
	if (sit == shm._sections.end())              // 如果不存在，返回成功（已经删除）
		return true;

	uint32_t idx = sit->second._index;           // 获取节在数组中的索引
	shm._sections.erase(sit);                    // 从索引映射表中移除节
	shm._block->_sections[idx]._state = 2;       // 将节的state标记为2（已删除）
	shm._block->_updatetime = TimeUtils::getLocalTimeNow();  // 更新块的修改时间为当前时间
	return true;                                 // 返回成功
}
```

## 键值对分配

### 分配字符串类型的键值对 allocate_string
```cpp
/**
 * @brief 分配字符串类型键值对的实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param initVal 初始值
 * @param bForceWrite 是否强制写入
 * @return 返回字符串指针
 * 
 * 分配流程：
 * 1. 调用make_valid创建或获取键值对
 * 2. 如果键是新分配的（type为0）或强制写入，使用初始值填充
 * 3. 返回字符串指针
 */
const char* ShareBlocks::allocate_string(const char* domain, const char* section, const char* key, const char* initVal /* = "" */, bool bForceWrite/* = false*/)
{
	SecInfo* secInfo = nullptr;                  // 节信息指针（输出参数）
	KeyInfo* keyInfo = (KeyInfo*)make_valid(domain, section, key, SMVT_STRING, secInfo);  // 创建或获取键值对
	if (keyInfo == nullptr)                      // 如果失败，返回NULL
		return NULL;

	if (keyInfo->_type == 0 || bForceWrite)      // 如果键是新分配的（type为0）或强制写入
	{
		//如果type为0，说明是新分配的，则用初始值填充
		keyInfo->_type = SMVT_STRING;            // 设置键的类型为字符串
		wt_strcpy(secInfo->_data + keyInfo->_offset, initVal, SMVT_SIZES[SMVT_STRING]);  // 复制初始值到数据区域（最多64字节）
	}

	return (secInfo->_data + keyInfo->_offset);  // 返回字符串指针
}
```

### 分配int32类型的键值对 allocate_int32
```cpp
```

### 分配int64类型的键值对 allocate_int64
```cpp
```

### 分配uint32类型的键值对 allocate_uint32
```cpp
```

### 分配uint64类型的键值对 allocate_uint64
```cpp
```

### 分配double类型的键值对 allocate_double
```cpp
```

## 键值对设置

### 设置字符串值 set_string
```cpp
/**
 * @brief 设置字符串值的实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param val 值
 * @return 返回是否设置成功
 * 
 * 设置流程：
 * 1. 调用make_valid创建或获取键值对
 * 2. 设置键的类型
 * 3. 复制值到数据区域
 */
bool ShareBlocks::set_string(const char* domain, const char* section, const char* key, const char* val)
{
	SecInfo* secInfo = nullptr;                  // 节信息指针（输出参数）
	KeyInfo* keyInfo = (KeyInfo*)make_valid(domain, section, key, SMVT_STRING, secInfo);  // 创建或获取键值对
	if (keyInfo == nullptr)                      // 如果失败，返回false
		return false;

	keyInfo->_type = SMVT_STRING;                // 设置键的类型为字符串
	wt_strcpy(secInfo->_data + keyInfo->_offset, val, SMVT_SIZES[SMVT_STRING]);  // 复制值到数据区域（最多64字节）

	return true;                                 // 返回成功
}
```

### 设置int32值 set_int32
```cpp
```

### 设置int64值 set_int64
```cpp
```

### 设置uint32值 set_uint32
```cpp
```

### 设置uint64值 set_uint64
```cpp
```

### 设置double值 set_double
```cpp
```

## 键值对获取

### 获取字符串值 get_string
```cpp
/**
 * @brief 获取字符串值的实现
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param defVal 默认值
 * @return 返回字符串指针
 * 
 * 获取流程：
 * 1. 调用check_valid检查键值对是否存在且类型匹配
 * 2. 如果不存在，返回默认值
 * 3. 返回字符串指针
 */
const char* ShareBlocks::get_string(const char* domain, const char* section, const char* key, const char* defVal /* = "" */)
{
	SecInfo* secInfo = nullptr;                  // 节信息指针（输出参数）
	KeyInfo* keyInfo = (KeyInfo*)check_valid(domain, section, key, SMVT_STRING, secInfo);  // 检查键值对是否有效
	if (keyInfo == nullptr)                      // 如果不存在或类型不匹配，返回默认值
		return defVal;

	return (const char*)(secInfo->_data + keyInfo->_offset);  // 返回字符串指针
}
```

### 获取int32值 get_int32
```cpp
```

### 获取int64值 get_int64
```cpp
```

### 获取uint32值 get_uint32
```cpp
```

### 获取uint64值 get_uint64
```cpp
```

### 获取double值 get_double
```cpp
```

## 命令队列管理
这套机制实现了一个**基于共享内存的单向命令队列（IPC Command Queue）**。
* **核心模型**：**单生产者（Commander）** -> **多消费者（Listener）**。
* **典型场景**：外部控制程序（如 GUI、Monitor）向策略引擎发送控制指令（如开始、暂停、调整参数）。

### 初始化命令队列 init_cmder
初始化命令队列的共享内存结构。
* 负责创建或打开共享文件，并初始化环形缓冲区的头指针。
* **写权限锁定**：通过记录进程 ID (`_cmdpid`)，确保只有创建该队列的进程（Commander）才有权写入指令。

**过程**：
* **检查初始化状态**：
  * 若已初始化即 `cmdPair = _cmd_blocks[name]._block` 非 NULL，直接返回 `true`。
* **物理文件准备**：
  * 确定文件名（默认为 `.cmd`）。
  * 若文件不存在，创建并截断大小为 `sizeof(CmdBlock)`（包含头部信息 + 128 个命令槽位）。
* **内存映射与结构初始化**：
  * 映射文件到内存 `cmdPair._domain`，获取其地址给 `cmdPair._block`
  * **新文件初始化**：若 `_capacity` 为 0，使用 `placement new` 原地构造 `CmdBlock`，初始化读写指针。
  * **身份绑定**：如果当前是命令下达者 (`isCmder == true`)，将共享内存中的 `cmdPair._block->_cmdpid` 设置为当前进程 ID。这是后续 `add_cmd` 鉴权的依据。
* **指针修正**：
  * `cmdPair._block->_writable %= cmdPair._block->_capacity`：防止上次异常退出导致索引溢出。
  * **同步读写指针**：若 `_readable` 有效（即发生过写入）
    * 修正其取模 `cmdPair._block->_readable %= cmdPair._block->_capacity`
    * 若 `cmdPair._block->_readable > cmdPair._block->_writable`（可能因循环覆盖导致）
      * `cmdPair._block->_writable += cmdPair._block->_capacity`
      * 以保持逻辑上的线性增长关系，确保环形缓冲区的连续性。

```cpp
/**
 * @brief 初始化命令队列的实现
 * @param name 命令队列名称
 * @param isCmder 是否为命令下达者
 * @param path 文件路径
 * @return 返回初始化是否成功
 */
bool ShareBlocks::init_cmder(const char* name, bool isCmder /* = false */, const char* path /* = "" */)
```

### 添加命令 add_cmd
向命令队列中**追加**一条新指令。
* **角色限制**：仅允许初始化时指定为 `Cmder` 且进程 ID 匹配的进程调用。
* **机制**：无锁环形写入（Lock-free Circular Write）。

**过程**：
* **前置检查**：
  * 检查内存块 `_cmd_blocks[name]._block` 是否存在
  * **身份鉴权**：检查共享内存中的 `cmdPair._block->_cmdpid` 是否等于当前进程 ID。如果不匹配（防止其他进程乱发指令），返回 `false`。
* **写入过程（两步提交）**：
  1. **抢占位置**：`wIdx = _writable++`。原子递增写索引，获取当前可写入的逻辑序号。
  2. **计算槽位**：`realIdx = wIdx % cmdPair._block->_capacity`。
  3. **写入数据**：将指令字符串 `strcpy` 到对应槽位 `cmdPair._block->_commands[realIdx]._command` ，并将状态置为 0（未读）。
  4. **发布更新**：`_readable = wIdx`。
     * 更新 *最新可读水位* 指针。这一步操作告诉所有消费者：“有一个新指令已经写完，现在的最新位置是 `wIdx`”。
```cpp
/**
 * @brief 添加命令的实现
 * @param name 命令队列名称
 * @param cmd 命令内容
 * @return 返回是否添加成功
 */
bool ShareBlocks::add_cmd(const char* name, const char* cmd)
```

### 获取命令 get_cmd
消费者（Listener）获取下一条未读指令。
* **身份过滤**：
  * 如果 `cmdPair._cmder`（当前进程是指令发出者），直接返回空字符串（自己不读自己的指令）。
* **状态机处理（根据 `lastIdx` 和全局 `_readable` 的关系）**：
  * **场景 A：队列未激活** (`_readable == UINT32_MAX`)
    * 说明 Commander 还没启动或没发过指令。
    * **动作**：将 `lastIdx` 设为魔数 `999999`（标记为“等待启动”），返回空。
  * **场景 B：初次连接** (`lastIdx == UINT32_MAX`)
    * 消费者刚启动，但队列里可能已经有一堆旧指令了。
    * **动作**：`lastIdx = _readable`。直接跳到最新位置，**忽略历史指令**，只听以后的。返回空。
  * **场景 C：刚从等待中恢复** (`lastIdx == 999999`)
    * 之前在场景 A 等待，现在 Commander 终于发指令了（`_readable` 变了）。
    * **动作**：`lastIdx = 0`。重置游标，准备读取第一条指令。
    * **返回**：索引为 0 的指令内容。
  * **场景 D：已读完所有指令** (`lastIdx >= _readable`)
    * 本地游标追上了全局水位。
    * **动作**：无新指令，返回空。
  * **场景 E：有新指令** (默认分支)
    * 全局水位 `_readable` 跑到了 `lastIdx` 前面。
    * **动作**：
      1. `lastIdx++`（游标前进一步）。
      2. 取模计算槽位。
      3. **返回**：该位置的指令内容。
```cpp
/**
 * @brief 获取命令的实现
 * @param name 命令队列名称
 * @param lastIdx 上次读取的索引（引用，用于跟踪读取位置）
 * @return 返回命令内容，如果没有新命令返回空字符串
 * 
 * 获取流程：
 * 1. 查找命令块配对
 * 2. 检查是否为命令下达者（下达者不需要获取命令）
 * 3. 处理各种索引状态：
 *    - 如果可读索引为UINT32_MAX（未初始化），设置lastIdx为999999并返回空
 *    - 如果lastIdx为UINT32_MAX（首次调用），设置为当前可读索引并返回空
 *    - 如果lastIdx为999999（刚启动），重置为0并返回第一条命令
 *    - 如果lastIdx大于等于可读索引（已读完），返回空
 *    - 否则，递增lastIdx并返回下一条命令
 */
const char* ShareBlocks::get_cmd(const char* name, uint32_t& lastIdx)
```

## 内部方法

### 创建或验证键值对 make_valid
创建或获取一个有效的键值对元数据指针（`KeyInfo*`）。
* **Master 专属特权**：如果在 Master 模式下调用且目标不存在，它会动态分配新的 Section（节）或 Key（键）。
* **资源管理**：负责检查内存块的物理限制（如最大节数、最大键数、数据区溢出）。
* **状态激活**：无论新建还是查找，都会将对应 Section 标记为有效。

**过程**：

1. **查找共享内存域**
   * 在 `_shm_blocks` 查找指定 `domain`。若不存在，直接返回 `nullptr`。
   * 根据传入的 `vType` 获取该数据类型所需的字节长度 `len = SMVT_SIZES[vType]`。
2. **查找或创建 Section（节）**
   * 在本地索引 `shm._sections` 中查找 `section` 名称。
   * **如果不存在（需要新建）**：
     * **权限检查**：若不是 Master (`!shm._master`)，无权创建，返回 `nullptr`。
     * **节容量检查**：若当前节数量已达上限 `MAX_SEC_CNT` (64)，无法分配，返回 `nullptr`。
     * **执行分配**：
       * 在 `_block->_sections` 数组末尾获取新位置。
       * 复制节名称，记录创建时间 `_updatetime`。
       * 在本地索引建立映射。
       * 增加块的计数 `_block->_count++`。
   * **如果已存在**：直接获取对应的 `kvPair`。
3. **激活 Section 状态**
   * 获取 Section 的物理指针 `secInfo = &shm._block->_sections[kvPair->_index]`。
   * **强制标记生效**：`secInfo->_state = 1`。即便以前被标记无效，现在访问它也会将其激活。
4. **查找或创建 Key（键）**
   * 在节的键索引 `kvPair->_keys` 中查找 `key` 名称。
   * **如果不存在（需要新建）**：
     * **权限检查**：若不是 Master，无权创建，返回 `nullptr`。
     * **Key 容量检查**：若该节内键数量 `shm._block->_count` 已达上限 `MAX_KEY_CNT` (64)，返回 `nullptr`。
     * **数据区空间检查**：若当前数据偏移 _offset + 新数据长度 len > 1024 字节，空间不足，返回 nullptr
     * **执行分配**：
       * 在 `secInfo->_keys` 数组末尾获取新位置。
       * 复制键名称，记录创建时间，设置**数据偏移量** `_offset`。
       * 将新 Key 加入本地索引。
       * 增加节的键计数 `secInfo->_count++`。
       * **推进偏移量**：`secInfo->_offset += len`，为下一个键预留空间。
   * **如果已存在**：直接获取对应的 `keyInfo` 指针。
5. **返回结果**
   * 返回指向共享内存中该 Key 元数据的指针 `KeyInfo*`。
```cpp
/**
 * @brief 创建或验证键值对的实现（内部方法）
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param vType 值类型
 * @param secInfo 节信息指针（输出参数）
 * @return 返回KeyInfo指针，如果失败返回nullptr
 */
void* ShareBlocks::make_valid(const char* domain, const char* section, const char* key, ValueType vType, SecInfo* &secInfo)
```

### 检查键值对是否有效 check_valid
检查键值对是否存在且类型匹配（严格只读）。
* **安全校验**：不进行任何内存分配或修改。
* **类型安全**：额外检查请求的数据类型 `vType` 与实际存储的 `_type` 是否一致，防止解释错误数据。

**过程**：
1. **查找共享内存域**
   * 在 `_shm_blocks` 查找 `domain`。若不存在，返回 `nullptr`。
2. **查找 Section（节）**
   * 在本地索引 `shm._sections` 中查找 `section`。
   * **若不存在**：直接返回 `nullptr`（读模式下不自动创建）。
   * 若存在，获取其物理指针并赋值给输出参数 `secInfo`。
3. **查找 Key（键）**
   * 在节的键索引 `kvPair->_keys` 中查找 `key`。
   * **若不存在**：直接返回 `nullptr`。
4. **类型校验（关键步骤）**
   * 获取找到的 `keyInfo` 指针。
   * **匹配类型**：检查 `keyInfo->_type` 是否等于传入的参数 `vType`。
     * 例如：如果试图用 `get_int32` 读取一个实际存储为 `string` 的键，这里会发现类型不匹配。
   * **若不匹配**：返回 `nullptr`。
5. **返回结果**
   * 校验通过，返回 `KeyInfo*` 指针。
```cpp
/**
 * @brief 检查键值对是否有效的实现（内部方法）
 * @param domain 域名称
 * @param section 节名称
 * @param key 键名称
 * @param vType 值类型
 * @param secInfo 节信息指针（输出参数）
 * @return 返回KeyInfo指针，如果不存在或类型不匹配返回nullptr
 */
void* ShareBlocks::check_valid(const char* domain, const char* section, const char* key, ValueType vType, SecInfo* &secInfo)
```